In [49]:
import numpy as np
import pandas as pd

<h1>Feature Scaling: Normalización y Estandarización</h1>

<h1>Dataset</h1>

<p>Se genera un dataset artificial de <b>m = 10</b> personas con 3 features de escalas muy distintas:</p>
<ul>
  <li><b>Edad:</b> años, entre 18 y 65</li>
  <li><b>Salario:</b> salario mensual en pesos ($), alrededor de 25,000</li>
  <li><b>Horas:</b> horas trabajadas por semana, entre 20 y 60</li>
</ul>

In [29]:
m = 10

rng = np.random.default_rng(seed=42)

edad = rng.integers(18, 66, m)
salario = rng.normal(25000, 8000, m).round()
horas = rng.integers(20, 61, m)

In [30]:
nombres = ['Edad', 'Salario', 'Horas']

datos = np.column_stack((edad, salario, horas)).tolist() #Se convierte a lista de listas para la implementación propia
datos

[[22.0, 14583.0, 38.0],
 [55.0, 26023.0, 29.0],
 [49.0, 22470.0, 23.0],
 [39.0, 24866.0, 42.0],
 [38.0, 18176.0, 56.0],
 [59.0, 32035.0, 22.0],
 [22.0, 31222.0, 55.0],
 [51.0, 25528.0, 53.0],
 [27.0, 34018.0, 31.0],
 [22.0, 28740.0, 45.0]]

In [31]:
print(type(datos))
print(type(datos[0]))

<class 'list'>
<class 'list'>


In [32]:
df = pd.DataFrame(datos, columns=nombres) #Convierte la lista de listas en una tabla; cada lista interna es una fila y "columns" les pone nombre a las columnas
df

,Edad,Salario,Horas
0,22.0,14583.0,38.0
1,55.0,26023.0,29.0
2,49.0,22470.0,23.0
3,39.0,24866.0,42.0
4,38.0,18176.0,56.0
5,59.0,32035.0,22.0
6,22.0,31222.0,55.0
7,51.0,25528.0,53.0
8,27.0,34018.0,31.0
9,22.0,28740.0,45.0


<p><b>Observación:</b> el salario está en el orden de decenas de miles, mientras que la edad y las horas están en decenas. Sin escalar, el salario dominaría cualquier cálculo de distancias o de gradientes.</p>

<h1>Implementación propia</h1>

In [33]:
def obtener_columna(datos, j):
    return [fila[j] for fila in datos] #List comprehension: toma el elemento j de cada fila

In [34]:
obtener_columna(datos, 0) #Columna de Edad al ser la primera

[22.0, 55.0, 49.0, 39.0, 38.0, 59.0, 22.0, 51.0, 27.0, 22.0]

<h2>Normalización (Min-Max Scaling)</h2>

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

<p>Lleva cada feature al rango [0, 1]: el valor mínimo se vuelve 0 y el máximo se vuelve 1.</p>
<p><b>Caso borde:</b> si todos los valores de la columna son iguales, x<sub>max</sub> − x<sub>min</sub> = 0 (división entre cero). En ese caso se asigna 0.</p>

In [51]:
def normalizar(datos):
    n_filas = len(datos)
    n_columnas = len(datos[0])

    resultado = [[0.0] * n_columnas for _ in range(n_filas)] #Se crea tabla de ceros del mismo tamaño que datos

    for j in range(n_columnas): 
        columna = obtener_columna(datos, j)
        x_min = min(columna)
        x_max = max(columna)
        rango = x_max - x_min #Se calcula el rango restando el mínimo al máximo de cada columna

        for i in range(n_filas): #Se normaliza cada valor siguiendo la fórmula
            resultado[i][j] = (datos[i][j] - x_min) / rango if rango != 0 else 0.0

    return resultado

<h2>Estandarización (Z-score)</h2>

$$z = \frac{x - \mu}{\sigma}$$

<p>Transforma cada feature para que tenga <b>media 0</b> y <b>desviación estándar 1</b>. El resultado no queda acotado a un rango fijo.</p>
<p><b>Caso borde:</b> si σ = 0 (todos los valores iguales) se asigna 0.</p>

<h3>Media y desviación estándar</h3>
<p>Para estandarizar se necesitan la media (μ) y la desviación estándar (σ) de cada columna, así que primero se definen estas funciones.</p>

$$\mu = \frac{1}{n}\sum_{i=1}^{n} x_i \qquad\qquad \sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \mu)^2}$$

<p>La desviación estándar se divide entre <b>n</b> (desviación poblacional), que es la que usa <code>StandardScaler</code>.</p>

In [36]:
def media(valores):
    return sum(valores) / len(valores)


def desviacion_estandar(valores):
    mu = media(valores) # μ
    suma_cuadrados = sum([(x - mu)**2 for x in valores])
    return (suma_cuadrados / len(valores)) ** 0.5 #** 0.5 es la raíz cuadrada

In [37]:
col_edad = obtener_columna(datos, 0)

print(f'media = {media(col_edad)}')
print(f'sigma = {desviacion_estandar(col_edad)}')

media = 38.4
sigma = 13.8144851514633


In [38]:
def estandarizar(datos):
    n_filas = len(datos)
    n_columnas = len(datos[0])

    resultado = [[0.0] * n_columnas for _ in range(n_filas)] #Se crea tabla de ceros del mismo tamaño que datos

    for j in range(n_columnas):
        columna = obtener_columna(datos, j)
        mu = media(columna) #Se calcula la media (μ) de cada columna
        sigma = desviacion_estandar(columna) #Se calcula la desviación estándar (σ) de cada columna

        for i in range(n_filas): #Se estandariza cada valor siguiendo la fórmula
            resultado[i][j] = (datos[i][j] - mu) / sigma if sigma != 0 else 0.0

    return resultado

<h2>Aplicando la implementación propia</h2>

In [39]:
datos_norm = normalizar(datos)
datos_std = estandarizar(datos)

In [40]:
pd.DataFrame(datos_norm, columns=nombres).round(4)

,Edad,Salario,Horas
0,0.0000,0.0000,0.4706
1,0.8919,0.5886,0.2059
2,0.7297,0.4058,0.0294
3,0.4595,0.5291,0.5882
4,0.4324,0.1849,1.0000
5,1.0000,0.8980,0.0000
6,0.0000,0.8561,0.9706
7,0.7838,0.5632,0.9118
8,0.1351,1.0000,0.2647
9,0.0000,0.7284,0.6765


In [41]:
pd.DataFrame(datos_std, columns=nombres).round(4)

,Edad,Salario,Horas
0,-1.1872,-1.9166,-0.1145
1,1.2016,0.0440,-0.8507
2,0.7673,-0.5649,-1.3416
3,0.0434,-0.1543,0.2127
4,-0.0290,-1.3008,1.3579
5,1.4912,1.0744,-1.4234
6,-1.1872,0.9350,1.2761
7,0.9121,-0.0408,1.1125
8,-0.8252,1.4142,-0.6871
9,-1.1872,0.5097,0.4581


<h3>Verificación</h3>
<p>Si las funciones están bien hechas:</p>
<ul>
  <li>Normalización: min = 0 y max = 1 en cada columna.</li>
  <li>Estandarización: media ≈ 0 y σ ≈ 1 en cada columna.</li>
</ul>

In [42]:
for j in range(len(nombres)):
    col = obtener_columna(datos_norm, j)
    print(f'{nombres[j]:8} -> min = {min(col)}, max = {max(col)}')

Edad     -> min = 0.0, max = 1.0
Salario  -> min = 0.0, max = 1.0
Horas    -> min = 0.0, max = 1.0


In [43]:
for j in range(len(nombres)):
    col = obtener_columna(datos_std, j)
    print(f'{nombres[j]:8} -> media = {media(col):.4f}, sigma = {desviacion_estandar(col):.4f}')

Edad     -> media = 0.0000, sigma = 1.0000
Salario  -> media = 0.0000, sigma = 1.0000
Horas    -> media = 0.0000, sigma = 1.0000


<h1>Scikit-Learn</h1>

In [44]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

In [45]:
X = np.array(datos)

X_norm_sk = MinMaxScaler().fit_transform(X)
X_std_sk = StandardScaler().fit_transform(X)

In [46]:
pd.DataFrame(X_norm_sk, columns=nombres).round(4)

,Edad,Salario,Horas
0,0.0000,0.0000,0.4706
1,0.8919,0.5886,0.2059
2,0.7297,0.4058,0.0294
3,0.4595,0.5291,0.5882
4,0.4324,0.1849,1.0000
5,1.0000,0.8980,0.0000
6,0.0000,0.8561,0.9706
7,0.7838,0.5632,0.9118
8,0.1351,1.0000,0.2647
9,0.0000,0.7284,0.6765


In [47]:
pd.DataFrame(X_std_sk, columns=nombres).round(4)

,Edad,Salario,Horas
0,-1.1872,-1.9166,-0.1145
1,1.2016,0.0440,-0.8507
2,0.7673,-0.5649,-1.3416
3,0.0434,-0.1543,0.2127
4,-0.0290,-1.3008,1.3579
5,1.4912,1.0744,-1.4234
6,-1.1872,0.9350,1.2761
7,0.9121,-0.0408,1.1125
8,-0.8252,1.4142,-0.6871
9,-1.1872,0.5097,0.4581


<h1>Comparación</h1>

In [50]:
print(f'Diferencia máxima (Normalización)   = {np.max(np.abs(np.array(datos_norm) - X_norm_sk))}')
print(f'Diferencia máxima (Estandarización) = {np.max(np.abs(np.array(datos_std) - X_std_sk))}')

Diferencia máxima (Normalización)   = 1.1102230246251565e-16
Diferencia máxima (Estandarización) = 0.0


<p><b>Resultado:</b> Al comparar el absoluto de las diferencias del método propio y el de Scikit-Learn, vemos que ambas implementaciones dan prácticamente los mismos valores. La diferencia en la normalización es minúscula (1.1102e-16), que en la práctica sería lo mismo.</p>